# Data Sanity and Model Diagnostics

This notebook is a post-modeling validation checkpoint. It checks that the modeling dataset, train/test split, feature lists, and model outputs are coherent before interpreting final performance.

It focuses on the current setup where `event_observed = 1` for all rows, so the main model-assumption checks are for ICU LOS prediction and log-LOS regression behavior rather than censored-survival assumptions.

## Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import numpy as np
import pandas as pd

from config import PROCESSED_DATA_DIR

## Load Artifacts

In [2]:
model_output_dir = PROCESSED_DATA_DIR / "model_outputs"

required_paths = {
    "modeling_dataset": PROCESSED_DATA_DIR / "modeling_dataset.parquet",
    "ready_predictors": PROCESSED_DATA_DIR / "modeling_ready_predictor_columns.csv",
    "ready_numeric": PROCESSED_DATA_DIR / "modeling_ready_numeric_columns.csv",
    "ready_categorical": PROCESSED_DATA_DIR / "modeling_ready_categorical_columns.csv",
    "train_test_split": PROCESSED_DATA_DIR / "modeling_train_test_split.csv",
    "lognormal_aft_metrics": model_output_dir / "lognormal_aft_metrics.csv",
    "lognormal_aft_predictions": model_output_dir / "lognormal_aft_test_predictions.csv",
    "gb_metrics": model_output_dir / "gb_log_los_metrics.csv",
    "gb_predictions": model_output_dir / "gb_log_los_test_predictions.csv",
}

missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    missing_display = "\n".join(f"- {name}: {required_paths[name]}" for name in missing_paths)
    raise FileNotFoundError(f"Missing required artifacts:\n{missing_display}")

modeling_df = pd.read_parquet(required_paths["modeling_dataset"])
ready_predictor_cols = pd.read_csv(required_paths["ready_predictors"])["predictor_column"].tolist()
ready_numeric_cols = pd.read_csv(required_paths["ready_numeric"])["numeric_predictor_column"].tolist()
ready_categorical_cols = pd.read_csv(required_paths["ready_categorical"])["categorical_predictor_column"].tolist()
split_df = pd.read_csv(required_paths["train_test_split"])

lognormal_metrics_df = pd.read_csv(required_paths["lognormal_aft_metrics"])
lognormal_predictions_df = pd.read_csv(required_paths["lognormal_aft_predictions"])
gb_metrics_df = pd.read_csv(required_paths["gb_metrics"])
gb_predictions_df = pd.read_csv(required_paths["gb_predictions"])

print(f"Modeling dataset: {modeling_df.shape}")
print(f"Ready predictors: {len(ready_predictor_cols)}")
print(f"Split rows: {len(split_df)}")


Modeling dataset: (94444, 457)
Ready predictors: 443
Split rows: 94444


## Core Data Sanity Checks

In [3]:
sanity_checks = []

def add_check(check, passed, detail):
    sanity_checks.append({"check": check, "passed": bool(passed), "detail": detail})

add_check(
    "modeling_dataset_has_rows",
    len(modeling_df) > 0,
    f"rows={len(modeling_df):,}",
)
add_check(
    "stay_id_unique",
    not modeling_df["stay_id"].duplicated().any(),
    f"duplicate_stay_id={modeling_df['stay_id'].duplicated().sum():,}",
)
add_check(
    "duration_present_positive",
    modeling_df["duration"].notna().all() and modeling_df["duration"].gt(0).all(),
    f"missing={modeling_df['duration'].isna().sum():,}, non_positive={modeling_df['duration'].le(0).sum():,}",
)
add_check(
    "event_observed_binary",
    modeling_df["event_observed"].isin([0, 1]).all(),
    str(modeling_df["event_observed"].value_counts(dropna=False).to_dict()),
)
add_check(
    "split_has_same_rows_as_modeling",
    len(split_df) == len(modeling_df),
    f"split_rows={len(split_df):,}, modeling_rows={len(modeling_df):,}",
)
add_check(
    "split_stay_id_unique",
    not split_df["stay_id"].duplicated().any(),
    f"duplicate_split_stay_id={split_df['stay_id'].duplicated().sum():,}",
)

sanity_checks_df = pd.DataFrame(sanity_checks)
sanity_checks_df

,check,passed,detail
0,modeling_dataset_has_rows,True,"rows=94,444"
1,stay_id_unique,True,duplicate_stay_id=0
2,duration_present_positive,True,"missing=0, non_positive=0"
3,event_observed_binary,True,{1: 94444}
4,split_has_same_rows_as_modeling,True,"split_rows=94,444, modeling_rows=94,444"
5,split_stay_id_unique,True,duplicate_split_stay_id=0


In [4]:
if not sanity_checks_df["passed"].all():
    failed = sanity_checks_df[~sanity_checks_df["passed"]]
    raise ValueError(f"Core sanity checks failed:\n{failed}")

print("Core sanity checks passed")


Core sanity checks passed


## Split Integrity Checks

The split should be subject-level: no subject should appear in both train and test.

In [5]:
split_subject_df = split_df[["subject_id", "split"]].drop_duplicates()
subject_split_counts = split_subject_df.groupby("subject_id")["split"].nunique()
leaked_subject_count = subject_split_counts.gt(1).sum()

split_summary_df = (
    split_df
    .groupby("split")
    .agg(
        rows=("stay_id", "size"),
        subjects=("subject_id", "nunique"),
        median_duration=("duration", "median"),
        mean_duration=("duration", "mean"),
        p95_duration=("duration", lambda x: x.quantile(0.95)),
        event_rate=("event_observed", "mean"),
    )
    .reset_index()
)

print(f"Subjects appearing in both train and test: {leaked_subject_count}")
split_summary_df

Subjects appearing in both train and test: 0


,split,rows,subjects,median_duration,mean_duration,p95_duration,event_rate
0,test,19064,13071,1.975087,3.679404,12.758060,1.0
1,train,75380,52284,1.962668,3.617537,12.659782,1.0


In [6]:
if leaked_subject_count > 0:
    raise ValueError("Subject leakage detected between train and test split")

print("Split integrity check passed")

Split integrity check passed


## Target Distribution Checks

In [7]:
duration_summary = modeling_df["duration"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995]
)
log_duration_summary = np.log1p(modeling_df["duration"]).describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.995]
)

print("Duration summary")
print(duration_summary)
print("\nlog1p(duration) summary")
print(log_duration_summary)


Duration summary
count    94444.000000
mean         3.630025
std          5.402474
min          0.001250
1%           0.209321
5%           0.543570
25%          1.096212
50%          1.965648
75%          3.862575
90%          7.920927
95%         12.681538
99%         26.438372
99.5%       33.986896
max        226.403079
Name: duration, dtype: float64

log1p(duration) summary
count    94444.000000
mean         1.237248
std          0.674866
min          0.001249
1%           0.190059
5%           0.434098
25%          0.740132
50%          1.087096
75%          1.581568
90%          2.188400
95%          2.616047
99%          3.311942
99.5%        3.554974
max          5.426724
Name: duration, dtype: float64


In [8]:
long_stay_thresholds = [7, 14, 30, 60]
long_stay_summary_df = pd.DataFrame(
    [
        {
            "threshold_days": threshold,
            "row_count": int(modeling_df["duration"].gt(threshold).sum()),
            "row_pct": modeling_df["duration"].gt(threshold).mean(),
        }
        for threshold in long_stay_thresholds
    ]
)

long_stay_summary_df

,threshold_days,row_count,row_pct
0,7,11171,0.118282
1,14,3897,0.041263
2,30,673,0.007126
3,60,58,0.000614


## Feature Sanity Checks

In [9]:
predictor_missing_df = pd.DataFrame(
    {
        "missing_count": modeling_df[ready_predictor_cols].isna().sum(),
        "missing_pct": modeling_df[ready_predictor_cols].isna().mean(),
        "dtype": modeling_df[ready_predictor_cols].dtypes.astype(str),
        "nunique_dropna": modeling_df[ready_predictor_cols].nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)

high_missing_predictors_df = predictor_missing_df[predictor_missing_df["missing_pct"].gt(0.80)]
constant_predictors_df = predictor_missing_df[predictor_missing_df["nunique_dropna"].le(1)]

print(f"Predictors with >80% missingness: {len(high_missing_predictors_df)}")
print(f"Constant predictors: {len(constant_predictors_df)}")
high_missing_predictors_df.head(20)

Predictors with >80% missingness: 0
Constant predictors: 0


,missing_count,missing_pct,dtype,nunique_dropna


In [10]:
leakage_exact_cols = {
    "los",
    "duration",
    "event_observed",
    "outtime",
    "intime",
    "admittime",
    "subject_id",
    "hadm_id",
    "stay_id",
}
leakage_patterns = ["outtime", "dischtime", "deathtime", "event_observed", "los"]

potential_leakage_cols = [
    col for col in ready_predictor_cols
    if col in leakage_exact_cols or any(pattern in col.lower() for pattern in leakage_patterns)
]

print(f"Potential leakage predictors: {len(potential_leakage_cols)}")
potential_leakage_cols

Potential leakage predictors: 0


[]

In [11]:
if potential_leakage_cols:
    raise ValueError(f"Potential leakage columns found: {potential_leakage_cols}")

print("Leakage check passed")

Leakage check passed


## Model Metrics Comparison

In [12]:
metrics_comparison_df = pd.concat(
    [
        lognormal_metrics_df.assign(source="lognormal_aft"),
        gb_metrics_df.assign(source="gradient_boosting_log_los"),
    ],
    ignore_index=True,
    sort=False,
)

metric_cols = [
    "source",
    "model",
    "train_c_index",
    "test_c_index",
    "train_mae_days",
    "test_mae_days",
    "train_median_ae_days",
    "test_median_ae_days",
    "train_rmse_days",
    "test_rmse_days",
    "train_median_pred_days",
    "test_median_pred_days",
]

metrics_comparison_df[metric_cols]

,source,model,train_c_index,test_c_index,train_mae_days,test_mae_days,train_median_ae_days,test_median_ae_days,train_rmse_days,test_rmse_days,train_median_pred_days,test_median_pred_days
0,lognormal_aft,lognormal_aft,0.736249,0.737534,2.104418,2.157047,0.813079,0.818840,4.976568,5.124627,2.044632,2.053978
1,gradient_boosting_log_los,hist_gradient_boosting_log_los,0.779661,0.761486,1.840963,2.038266,0.783363,0.841811,4.404514,4.901499,2.222473,2.248711


## Prediction Tail Checks

In [13]:
def prediction_tail_summary(predictions_df, prediction_col, model_name):
    return {
        "model": model_name,
        "observed_max_days": predictions_df["duration"].max(),
        "predicted_max_days": predictions_df[prediction_col].max(),
        "predicted_99th_pct_days": predictions_df[prediction_col].quantile(0.99),
        "predicted_999th_pct_days": predictions_df[prediction_col].quantile(0.999),
        "pred_over_30_days_count": int(predictions_df[prediction_col].gt(30).sum()),
        "pred_over_60_days_count": int(predictions_df[prediction_col].gt(60).sum()),
    }

prediction_tail_check_df = pd.DataFrame(
    [
        prediction_tail_summary(lognormal_predictions_df, "predicted_median_los_days", "lognormal_aft"),
        prediction_tail_summary(gb_predictions_df, "predicted_los_days", "gradient_boosting_log_los"),
    ]
)

prediction_tail_check_df

,model,observed_max_days,predicted_max_days,predicted_99th_pct_days,predicted_999th_pct_days,pred_over_30_days_count,pred_over_60_days_count
0,lognormal_aft,121.303368,46.956875,10.783367,22.008449,6,0
1,gradient_boosting_log_los,121.303368,27.133458,9.945802,16.374322,0,0


## Residual Diagnostics

For log-LOS models, residuals should be reasonably centered. Long-stay underprediction is expected and should be quantified separately.

In [14]:
gb_residuals_df = gb_predictions_df.copy()
gb_residuals_df["residual_days"] = gb_residuals_df["duration"] - gb_residuals_df["predicted_los_days"]
gb_residuals_df["log_residual"] = np.log1p(gb_residuals_df["duration"]) - np.log1p(gb_residuals_df["predicted_los_days"])

aft_residuals_df = lognormal_predictions_df.copy()
aft_residuals_df["residual_days"] = aft_residuals_df["duration"] - aft_residuals_df["predicted_median_los_days"]
aft_residuals_df["log_residual"] = np.log1p(aft_residuals_df["duration"]) - np.log1p(aft_residuals_df["predicted_median_los_days"])

residual_summary_df = pd.DataFrame(
    [
        {
            "model": "lognormal_aft",
            "mean_residual_days": aft_residuals_df["residual_days"].mean(),
            "median_residual_days": aft_residuals_df["residual_days"].median(),
            "mean_log_residual": aft_residuals_df["log_residual"].mean(),
            "median_log_residual": aft_residuals_df["log_residual"].median(),
        },
        {
            "model": "gradient_boosting_log_los",
            "mean_residual_days": gb_residuals_df["residual_days"].mean(),
            "median_residual_days": gb_residuals_df["residual_days"].median(),
            "mean_log_residual": gb_residuals_df["log_residual"].mean(),
            "median_log_residual": gb_residuals_df["log_residual"].median(),
        },
    ]
)

residual_summary_df

,model,mean_residual_days,median_residual_days,mean_log_residual,median_log_residual
0,lognormal_aft,1.055213,-0.143348,0.060198,-0.057690
1,gradient_boosting_log_los,0.846220,-0.224837,0.005199,-0.086939


In [15]:
gb_error_by_observed_los_df = gb_residuals_df.copy()
gb_error_by_observed_los_df["observed_los_group"] = pd.cut(
    gb_error_by_observed_los_df["duration"],
    bins=[0, 1, 2, 4, 7, 14, 30, np.inf],
    labels=["<=1", "1-2", "2-4", "4-7", "7-14", "14-30", ">30"],
    include_lowest=True,
)

gb_error_by_observed_los_summary_df = (
    gb_error_by_observed_los_df
    .groupby("observed_los_group", observed=True)
    .agg(
        rows=("stay_id", "size"),
        median_observed_los=("duration", "median"),
        median_predicted_los=("predicted_los_days", "median"),
        mae_days=("absolute_error_days", "mean"),
        median_ae_days=("absolute_error_days", "median"),
        mean_residual_days=("residual_days", "mean"),
    )
    .reset_index()
)

gb_error_by_observed_los_summary_df

,observed_los_group,rows,median_observed_los,median_predicted_los,mae_days,median_ae_days,mean_residual_days
0,<=1,3907,0.737454,1.263299,0.666908,0.523361,-0.635931
1,1-2,5750,1.385799,1.946219,0.851291,0.579953,-0.764240
2,2-4,4737,2.782708,2.689716,1.080580,0.813499,-0.234319
3,4-7,2365,5.043785,3.500199,1.993900,1.859706,1.243415
4,7-14,1513,9.242951,4.544244,4.902136,4.863811,4.710012
5,14-30,651,18.327350,5.353367,13.567010,13.157347,13.488027
6,>30,141,38.836227,5.853761,37.401703,32.148518,37.401703


## Subgroup Error Checks

In [16]:
test_with_features_df = gb_predictions_df.merge(
    modeling_df[["stay_id", "age_group", "gender", "race_grouped", "careunit_group", "admission_urgency_group"]],
    on="stay_id",
    how="left",
)

subgroup_cols = ["age_group", "gender", "race_grouped", "careunit_group", "admission_urgency_group"]
subgroup_summaries = []

for subgroup_col in subgroup_cols:
    subgroup_summary = (
        test_with_features_df
        .groupby(subgroup_col, observed=True)
        .agg(
            rows=("stay_id", "size"),
            median_observed_los=("duration", "median"),
            median_predicted_los=("predicted_los_days", "median"),
            mae_days=("absolute_error_days", "mean"),
            median_ae_days=("absolute_error_days", "median"),
        )
        .reset_index()
        .rename(columns={subgroup_col: "subgroup_value"})
    )
    subgroup_summary["subgroup"] = subgroup_col
    subgroup_summaries.append(subgroup_summary)

subgroup_error_df = pd.concat(subgroup_summaries, ignore_index=True)
subgroup_error_df.sort_values(["mae_days", "rows"], ascending=[False, False]).head(30)

,subgroup_value,rows,median_observed_los,median_predicted_los,mae_days,median_ae_days,subgroup
14,NEURO,1396,2.986487,3.060646,2.592874,1.283159,careunit_group
10,UNKNOWN,1705,2.361551,2.748821,2.578442,1.060520,race_grouped
20,URGENT,3073,2.311493,2.567616,2.452941,1.066053,admission_urgency_group
15,OTHER,33,3.005984,3.060780,2.429766,0.807056,careunit_group
16,SURGICAL,5122,1.938947,2.283491,2.362434,0.896209,careunit_group
19,OBSERVATION,3033,2.013623,2.405021,2.218426,0.929263,admission_urgency_group
8,HISPANIC,687,2.012593,2.255612,2.195904,0.931965,race_grouped
0,18-39,1939,1.763194,2.022396,2.165913,0.744517,age_group
5,M,10670,1.998229,2.284557,2.136384,0.860663,gender
1,40-59,5307,1.934745,2.268883,2.110848,0.853775,age_group


## Assumption and Readiness Summary

In [17]:
assumption_summary = []

assumption_summary.append({
    "check": "all_events_observed",
    "status": "note",
    "detail": "event_observed is all 1; models are LOS prediction models, not censored-survival endpoint models.",
})
assumption_summary.append({
    "check": "subject_level_split",
    "status": "pass" if leaked_subject_count == 0 else "fail",
    "detail": f"subjects in both train and test: {leaked_subject_count}",
})
assumption_summary.append({
    "check": "leakage_predictors_absent",
    "status": "pass" if len(potential_leakage_cols) == 0 else "fail",
    "detail": f"potential leakage predictors: {len(potential_leakage_cols)}",
})
assumption_summary.append({
    "check": "raw_count_features_excluded_in_final_models",
    "status": "pass",
    "detail": "notebook 12 and 13 exclude *_count_24h predictors to avoid charting-density leverage.",
})
assumption_summary.append({
    "check": "gradient_boosting_improves_baseline",
    "status": "pass" if gb_metrics_df.loc[0, "test_mae_days"] < lognormal_metrics_df.loc[0, "test_mae_days"] else "review",
    "detail": f"GB test MAE={gb_metrics_df.loc[0, 'test_mae_days']:.3f}; AFT test MAE={lognormal_metrics_df.loc[0, 'test_mae_days']:.3f}",
})
assumption_summary.append({
    "check": "long_stay_underprediction",
    "status": "review",
    "detail": "Gradient boosting underpredicts rare very-long stays; inspect error by observed LOS group.",
})

assumption_summary_df = pd.DataFrame(assumption_summary)
assumption_summary_df

,check,status,detail
0,all_events_observed,note,event_observed is all 1; models are LOS predic...
1,subject_level_split,pass,subjects in both train and test: 0
2,leakage_predictors_absent,pass,potential leakage predictors: 0
3,raw_count_features_excluded_in_final_models,pass,notebook 12 and 13 exclude *_count_24h predict...
4,gradient_boosting_improves_baseline,pass,GB test MAE=2.038; AFT test MAE=2.157
5,long_stay_underprediction,review,Gradient boosting underpredicts rare very-long...


## Save Diagnostic Outputs

In [18]:
diagnostic_output_dir = PROCESSED_DATA_DIR / "model_outputs"
diagnostic_output_dir.mkdir(exist_ok=True)

sanity_checks_df.to_csv(diagnostic_output_dir / "diagnostic_core_sanity_checks.csv", index=False)
split_summary_df.to_csv(diagnostic_output_dir / "diagnostic_split_summary.csv", index=False)
long_stay_summary_df.to_csv(diagnostic_output_dir / "diagnostic_long_stay_summary.csv", index=False)
predictor_missing_df.to_csv(diagnostic_output_dir / "diagnostic_predictor_missingness.csv")
metrics_comparison_df.to_csv(diagnostic_output_dir / "diagnostic_model_metrics_comparison.csv", index=False)
prediction_tail_check_df.to_csv(diagnostic_output_dir / "diagnostic_prediction_tail_check.csv", index=False)
residual_summary_df.to_csv(diagnostic_output_dir / "diagnostic_residual_summary.csv", index=False)
gb_error_by_observed_los_summary_df.to_csv(diagnostic_output_dir / "diagnostic_gb_error_by_observed_los.csv", index=False)
subgroup_error_df.to_csv(diagnostic_output_dir / "diagnostic_gb_subgroup_error.csv", index=False)
assumption_summary_df.to_csv(diagnostic_output_dir / "diagnostic_assumption_summary.csv", index=False)

print(f"Saved diagnostics to {diagnostic_output_dir}")

Saved diagnostics to /Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/model_outputs
